# 03 · Guasti

Quanto si vede ogni guasto nelle metriche e se il servizio colpito emerge. Per ogni run di guasto (test e dev) i segnali sono confrontati con il settle: lo score di un servizio è il massimo |z robusto| sulle sue feature nei primi `AFTER` secondi dopo l'iniezione, come il RobustScorer di BARO. Non è il modello: è il minimo che il VAE deve superare.

In fondo lo stesso score calcolato ai cambi di carico delle sessioni pulite: è la misura del bias. Se i cambi di carico danno score simili a quelli dei guasti, un detector li confonde.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)

In [ ]:
DATA = Path("/home/tecnico/datasets/pilot")
STEP = 30
AFTER = 300  # secondi dopo l'iniezione su cui si calcola lo score
NEAR = 60    # un cambio di carico entro NEAR secondi dall'iniezione rende la run "vicina"

loaded = {k: ex.load(a, STEP) for k, a in ex.find_runs(DATA).items()}
faults = {k: r for k, r in loaded.items() if r.kind in ex.FAULT_KINDS}
clean = {k: r for k, r in loaded.items() if r.kind == "clean"}
print(len(faults), "run di guasto,", len(clean), "sessioni pulite")

## Visibilità e rank del target

Una riga per run. `rank target` = posizione del servizio colpito nello score (1 = primo); `primi 3` = servizi con lo score più alto, con la feature che lo determina; `gap carico s` = distanza tra iniezione e cambio di carico più vicino. Le run noop hanno solo lo score massimo, che è un falso allarme.

In [ ]:
rows = []
for name, run in faults.items():
    scored = ex.score_services(run, AFTER)
    order = [s for s, _, _ in scored]
    targets = ex.target_services(run)
    rows.append({
        "run": name,
        "guasto": run.labels["fault_type"],
        "target": ", ".join(targets) or "-",
        "rank target": min((order.index(t) + 1 for t in targets), default=np.nan),
        "score target": max((v for s, v, _ in scored if s in targets), default=np.nan),
        "score max": scored[0][1],
        "primi 3": ", ".join(f"{s} {v:.0f} ({f})" for s, v, f in scored[:3]),
        "gap carico s": ex.gap_to_injection(run),
        "valida": all(v[0] for k, v in run.labels["checks"].items() if k != "window"),
    })
table = pd.DataFrame(rows)
table.round(1)

## Target contro gli altri servizi

Per ogni run: z robusto del target rispetto al settle (linee colorate) e banda tra 10° e 90° percentile degli altri servizi (grigio). Al massimo `MAX_RUNS` run.

In [ ]:
FEATURES = ["cpu", "mem_working_set", "rx_bytes", "threads", "restarts"]
MAX_RUNS = 30
for name, run in list(faults.items())[:MAX_RUNS]:
    targets = ex.target_services(run)
    if not targets:
        continue
    z = ex.robust_z(run.x, ex.settle_mask(run))
    others = [i for i, s in enumerate(run.services) if s not in targets]
    minutes = run.t_rel / 60
    fig, axes = plt.subplots(1, len(FEATURES), figsize=(18, 2.6), sharex=True)
    for a, feature in zip(axes, FEATURES):
        j = run.features.index(feature)
        low, high = np.nanpercentile(z[:, others, j], [10, 90], axis=1)
        a.fill_between(minutes, low, high, color="0.85")
        for target in targets:
            a.plot(minutes, z[:, run.services.index(target), j], label=target)
        a.axvline(0, color="red", ls="--")
        a.set_title(feature)
        a.set_ylim(-15, 15)
    axes[0].legend(fontsize=7)
    fig.suptitle(f"{name}: {run.labels['fault_type']}", fontsize=10)
    plt.show()

## Stress graduato contro aumento di carico

Scenari dev di stress CPU: aumento di CPU del servizio stressato (mediana dopo l'iniezione meno mediana del settle, in millicore), accanto agli aumenti che lo stesso servizio ha tra due livelli di rate consecutivi nelle sessioni pulite. Se lo stress più debole è più piccolo dei salti di carico, solo la relazione con il traffico lo può distinguere.

In [ ]:
rows = []
for name, run in faults.items():
    if "stress" not in name:
        continue
    service = ex.target_services(run)[0]
    cpu = ex.series(run, service, "cpu") * 1000
    after = (run.t_rel > 60) & (run.t_rel <= AFTER + 60)
    rows.append({"origine": name, "servizio": service, "aumento mC": np.nanmedian(cpu[after]) - np.nanmedian(cpu[ex.settle_mask(run)])})
for name, run in clean.items():
    levels = ex.by_level(run, "cpu").groupby(level="rate").median() * 1000
    for service in {r["servizio"] for r in rows}:
        if service in levels:
            for r0, r1 in zip(levels.index[:-1], levels.index[1:]):
                rows.append({"origine": f"{name}: da {r0} a {r1} req/s", "servizio": service,
                             "aumento mC": levels.loc[r1, service] - levels.loc[r0, service]})
if rows:
    display(pd.DataFrame(rows).round(1))
else:
    print("Nessuno scenario di stress in", DATA)

## Vicino o lontano da un cambio di carico

Rank e score del target nelle run con un cambio di carico entro `NEAR` secondi dall'iniezione e nelle altre.

In [ ]:
if len(table):
    table["vicina"] = table["gap carico s"] < NEAR
    display(table.groupby("vicina")[["rank target", "score target", "score max"]].describe().round(1).T)

## Controllo negativo: lo stesso score ai cambi di carico

Nelle sessioni pulite ogni cambio di rate fa da finta iniezione: riferimento i 5 minuti prima, score massimo sui servizi nei `AFTER` secondi dopo. A confronto con lo score del target nelle run di guasto.

In [ ]:
negative = []
for name, run in clean.items():
    times, steps = ex.load_changes(run)
    for time, (r0, r1) in zip(times, steps):
        ref = (run.t > time - 300) & (run.t <= time)
        after = (run.t > time) & (run.t <= time + AFTER)
        if ref.sum() >= 5 and after.sum() >= 5:
            z = np.abs(ex.robust_z(run.x, ref))
            negative.append({"sessione": name, "da": r0, "a": r1, "score max": float(np.nanmax(z[after]))})
negative = pd.DataFrame(negative)

fig, a = plt.subplots(figsize=(10, 3.5))
bins = np.logspace(0, 3, 30)
if len(negative):
    a.hist(negative["score max"].clip(1, 999), bins=bins, alpha=0.6, label=f"cambi di carico nelle sessioni pulite ({len(negative)})")
if len(table) and table["score target"].notna().any():
    a.hist(table["score target"].dropna().clip(1, 999), bins=bins, alpha=0.6, label=f"target nelle run di guasto ({table['score target'].notna().sum()})")
a.set_xscale("log")
a.set(xlabel="score (max |z robusto|)", ylabel="eventi")
a.legend()
plt.show()
if len(negative):
    display(negative.groupby(["da", "a"])["score max"].describe().round(1))